# Testing of random forest model transfer
This is a trial to test whether a model can be transferred from one classification purpose to other context

In [14]:
# !python -m pip install .. --quiet

import ee 
import luma_ge

# service_account_path = '../auth/ee-epstm2024.json'
# luma_ge.initialize_with_service_account(service_account_path)

ee.Authenticate()
ee.Initialize()

In [15]:
import geemap

region_name = "Sumatera"
regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

# Satellite image retrieval

In [16]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2018-01-01'
end = '2018-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True,
                                                            coverage_scale=100)

2026-07-03 14:10:19,105 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-07-03 14:10:19,106 - final_Image - INFO - final_Image creation initialized.
2026-07-03 14:10:19,107 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-07-03 14:10:19,108 - Reflectance_Data - INFO - Date range: 2018-01-01 to 2018-12-31
2026-07-03 14:10:19,109 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-07-03 14:10:19,110 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-07-03 14:10:19,111 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-07-03 14:10:19,111 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-07-03 14:10:20,278 - final_Image - INFO - Creating quality mosaic from 250 images using NDVI as quality metric
2026-07-03 14:10:20,280 - final_Image - INFO - Quality mosaic created covering AOI with best available pixels
20

# Classification scheme
Use RESTORE+ scheme

In [17]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

# Load default training data

In [18]:
from luma_ge.sample_data import SyncTrainData

TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=classification_df,
            aoi_geometry=aoi,
            training_ee_path=TrainEePath
        )

        # Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
# TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=False)
        
        # Check sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )

TrainDataFinal = TrainDataDict.get('training_data')

2026-07-03 14:10:30,343 - luma_ge.sample_data - INFO - Loading training data from EE asset: projects/ee-rg2icraf/assets/Indonesia_lulc_Sample
2026-07-03 14:10:31,538 - luma_ge.sample_data - INFO - Initial feature count: 1130000
2026-07-03 14:10:31,542 - luma_ge.sample_data - INFO - Filtering by AOI bounds...
2026-07-03 14:10:31,546 - luma_ge.sample_data - INFO - AOI geometry type: <class 'ee.geometry.Geometry'>
2026-07-03 14:10:35,233 - luma_ge.sample_data - INFO - Features after AOI filter: 170000
2026-07-03 14:10:35,234 - luma_ge.sample_data - INFO - Converting to GeoDataFrame...
2026-07-03 14:10:38,077 - luma_ge.sample_data - INFO - Collection size: 170000
2026-07-03 14:10:38,078 - luma_ge.sample_data - INFO - Collection has 170000 features, implementing stratified sampling for class representation
2026-07-03 14:10:40,649 - luma_ge.sample_data - INFO - Found 17 unique classes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
2026-07-03 14:10:40,650 - luma_ge.sample_data -

# Generate model classifier

In [19]:
from ee import classifier

from luma_ge.classification import FeatureExtraction

labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, median_landsat, 
                            class_prop='kelas', train_ratio=0.5)

# create classifier in multiprobability output mode
clf_sumatra_prob = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=1
).setOutputMode('MULTIPROBABILITY').train(
        features=strafied_train,
        classProperty='kelas',
        inputProperties=median_landsat.bandNames()
        )



2026-07-03 14:11:15,188 - pyogrio._io - INFO - Created 5,000 records


Stratified Random Split Training Pixel Size: 2454
Stratified Random Split Testing Pixel Size: 2546


# Save classifier to GEE asset

This feature is not available for `MULTIPROBABILITY` output mode.

In [20]:

# # save classifier
# assetId = 'projects/epistem-490415/assets/modular_mapping_sandbox/clf_sumatra_classification'

# # Create and start the export task
# task = ee.batch.Export.classifier.toAsset(
#   classifier=clf_sumatra,
#   description='clf_sumatra_classification',
#   assetId=assetId
# )
# task.start()

# check to load the classifier from asset

# savedClassifier = ee.Classifier.load(assetId).setOutputMode('MULTIPROBABILITY')


# Classify and save MULTIPROBABILITY image stack to Google Drive

In [23]:
# rename band names of the probability
import re


def sanitize_band_name(name: str) -> str:
    """GEE band names should avoid spaces/special chars for safety downstream."""
    name = str(name).strip()
    name = re.sub(r'[^\w]+', '_', name)   # replace non-word chars with underscore
    name = re.sub(r'_+', '_', name).strip('_')
    return name

classification_df = classification_df.sort_values("ID").reset_index(drop=True)
class_labels = [sanitize_band_name(name) for name in classification_df["Land Cover Class"]]

# Skip the export and just use the classifier directly
probability_stack = median_landsat.select(median_landsat.bandNames()).classify(clf_sumatra_prob)
probability_stack = probability_stack.arrayFlatten([class_labels])

# --- Export to Drive ---
task = ee.batch.Export.image.toDrive(
    image=probability_stack,
    description='sumatra_multiprobability_stack',
    folder='GEE_exports',
    fileNamePrefix='sumatra_multiprobability_stack',
    region=aoi,
    scale=100,
    crs='EPSG:4326',
    maxPixels=1e13,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)
task.start()

# Hard classification map visualization

In [ ]:
# # Create visualization parameters based on classification scheme
# ids = classification_df["ID"].tolist()
# class_names = classification_df["Land Cover Class"].tolist()
# colors = classification_df["Color Palette"].tolist()

# # Build visualization parameters with palette
# vis_params = {
#     "min": min(ids),
#     "max": max(ids),
#     "palette": colors  # Use colors from classification scheme
# }

# # Build legend dictionary for display
# legend_dict = {class_name: color for class_name, color in zip(class_names, colors)}

# Map = geemap.Map() 
# Map.centerObject(aoi, 7)
# Map.addLayer(classified_map, vis_params, "LULC Classification")

# # # === Add Legend ===
# Map.add_legend(
#     title="Land Cover Classification", 
#     legend_dict=legend_dict
#     )

# Map


# Probability Bands Visualization

In [ ]:
# Visualize individual probability bands
prob_map = geemap.Map()
prob_map.centerObject(aoi, 7)

# Get band names from prob_bands
band_names = prob_bands.bandNames().getInfo()
print(f"Available probability bands: {band_names}")

# Add each probability band to the map
prob_vis_params = {
    "min": 0,
    "max": 1,
    "palette": ['white', 'black']  # Grayscale for probability (0=white, 1=black)
}

for i, band_name in enumerate(band_names):
    band = prob_bands.select(band_name)
    prob_map.addLayer(band, prob_vis_params, f"Probability - {band_name}", shown=False)

# Also create an RGB composite using first 3 probability bands
if len(band_names) >= 3:
    rgb_composite = prob_bands.select(band_names[:3])
    prob_map.addLayer(rgb_composite, {"min": 0, "max": 1}, "RGB Composite (prob_0, prob_1, prob_2)", shown=True)

prob_map